# Stage 1: House Price Enrichment Ingestion Layer

This section prepares the UK House Price Index enrichment dataset for ingestion into Snowflake.  
The original dataset was too large and wide for direct upload, so it was processed locally in Python before being loaded into the Snowflake Bronze layer.

## 1. Import Required Libraries

The pipeline uses pandas for CSV processing and glob to locate the split source files.

## 2. Locate Split House Price Files

The original house price CSV was split into seven smaller files to support scalable batch processing and avoid upload limits.

## 3. Read Files in Batches

Each split CSV file is read iteratively rather than loading the full source file in one step.  
This supports scalable ingestion and reduces memory pressure.

## 4. Remove Empty and Unnecessary Columns

Completely empty columns are removed.  
LSOA-level fields are also dropped because the final reporting dataset does not require this level of geographic detail.

## 5. Clean House Price Values

House price values are cleaned by removing formatting issues, converting values to numeric format, and removing rows where no valid price is available.

## 6. Combine Cleaned Batches

All cleaned batches are combined into one dataframe. Duplicate rows are removed to improve data quality before aggregation.

## 7. Aggregate to Local Authority × Period Grain

The cleaned data is aggregated to local authority and period level.  
This creates a smaller, more suitable enrichment dataset for Snowflake ingestion and later police force mapping.

## 8. Export Bronze Ingestion File

The processed enrichment dataset is exported as a CSV file for upload into the Snowflake Bronze layer.

In [1]:
import pandas as pd
import glob

# Find all split CSV files
files = sorted(glob.glob("split_house_prices/house_prices_part_*.csv"))

print("Files found:", len(files))

clean_frames = []

for file in files:

    print("Reading:", file)

    df = pd.read_csv(file, dtype=str, low_memory=False)

    # Drop completely empty columns
    df = df.dropna(axis=1, how='all')

    # Drop unnecessary LSOA columns if they exist
    cols_to_drop = ["LSOA code", "LSOA name"]

    df = df.drop(
        columns=[c for c in cols_to_drop if c in df.columns],
        errors="ignore"
    )

    # Define identifier columns
    id_cols = ["Local authority code", "Local authority name"]

    # Keep only id columns + non-empty value columns
    value_cols = [c for c in df.columns if c not in id_cols]

    # Convert wide → long
    df_long = df.melt(
        id_vars=id_cols,
        value_vars=value_cols,
        var_name="period",
        value_name="average_house_price"
    )

    # Clean price values
    df_long["average_house_price"] = (
        df_long["average_house_price"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    # Convert to numeric
    df_long["average_house_price"] = pd.to_numeric(
        df_long["average_house_price"],
        errors="coerce"
    )

    # Remove null prices
    df_long = df_long.dropna(subset=["average_house_price"])

    clean_frames.append(df_long)

# Combine all cleaned chunks
house_clean = pd.concat(clean_frames, ignore_index=True)

# Final cleanup
house_clean = house_clean.drop_duplicates()

# Save cleaned dataset
house_clean.to_csv("house_prices_clean_long.csv", index=False)

print("Saved: house_prices_clean_long.csv")
print("Final rows:", len(house_clean))
print("Final columns:", len(house_clean.columns))
print(house_clean.head())

Files found: 7
Reading: split_house_prices/house_prices_part_1.csv
Reading: split_house_prices/house_prices_part_2.csv
Reading: split_house_prices/house_prices_part_3.csv
Reading: split_house_prices/house_prices_part_4.csv
Reading: split_house_prices/house_prices_part_5.csv
Reading: split_house_prices/house_prices_part_6.csv
Reading: split_house_prices/house_prices_part_7.csv
Saved: house_prices_clean_long.csv
Final rows: 3081559
Final columns: 4
  Local authority code Local authority name                period  \
0            E06000001           Hartlepool  Year ending Dec 1995   
1            E06000001           Hartlepool  Year ending Dec 1995   
2            E06000001           Hartlepool  Year ending Dec 1995   
3            E06000001           Hartlepool  Year ending Dec 1995   
4            E06000001           Hartlepool  Year ending Dec 1995   

   average_house_price  
0              34750.0  
1              25000.0  
2              27000.0  
3              44500.0  
4        

In [2]:
house_clean_la = (
    house_clean
    .groupby(
        ["Local authority code", "Local authority name", "period"],
        as_index=False
    )
    .agg(
        average_house_price=("average_house_price", "mean"),
        records_used=("average_house_price", "count")
    )
)

house_clean_la.to_csv("house_prices_local_authority_period.csv", index=False)

print("Saved: house_prices_local_authority_period.csv")
print("Rows:", len(house_clean_la))
print(house_clean_la.head())

Saved: house_prices_local_authority_period.csv
Rows: 36630
  Local authority code Local authority name                period  \
0            E06000001           Hartlepool  Year ending Dec 1995   
1            E06000001           Hartlepool  Year ending Dec 1996   
2            E06000001           Hartlepool  Year ending Dec 1997   
3            E06000001           Hartlepool  Year ending Dec 1998   
4            E06000001           Hartlepool  Year ending Dec 1999   

   average_house_price  records_used  
0         40984.519231            52  
1         43790.117647            51  
2         42786.979167            48  
3         44867.843137            51  
4         47151.226415            53  


- Local Jupyter pre-processing: raw 7 CSV chunks → reduced upload file